# 03 · When Molecular Similarity Fails: CYP Inhibition Activity Cliffs

**A self-contained, Google Colab-ready exploratory study using the public OpenADMET training data.**

**Research aims**

1. Identify structurally similar compounds with substantially different measured CYP inhibition.
2. Compare activity-cliff frequencies for CYP1A2, CYP2C9, CYP2D6, and CYP3A4.
3. Evaluate which apparent cliffs remain pronounced after considering experimental pIC50 confidence intervals.
4. Inspect representative molecular pairs and highlight structural differences.
5. **Without target leakage**, test whether activity variation among **training-only** nearest neighbors helps rank prediction errors on held-out scaffolds.

**Important distinction:** Whole-dataset activity cliffs describe *observed chemistry* retrospectively. The prospective validation experiment later in this notebook uses *only training-set activities* to build neighborhood features for validation molecules. We never use blinded challenge test labels.

This is exploratory research. Pairwise observations are highly dependent; raw pair counts are **not** independent samples or estimates of causal relationships.

## 1 · Install dependencies

Designed for a free Colab CPU runtime. The model-validation section is the slowest part; the default runs one scaffold split per isoform, with a small random-forest ensemble.

In [ ]:
%pip -q install "rdkit>=2023.9" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "seaborn>=0.12"

## 2 · Configuration, imports, and output directory

In [ ]:
import os, json, platform, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from rdkit import Chem, DataStructs, rdBase, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

RDLogger.DisableLog('rdApp.warning')
sns.set_theme(style='whitegrid', context='notebook')
ISOFORMS = ('CYP1A2','CYP2C9','CYP2D6','CYP3A4')
SEED = 2026
SIMILARITY_THRESHOLD = 0.75
DELTA_PIC50_THRESHOLD = 1.0  # 10-fold IC50 difference
PAIR_COLLECTION_MIN_SIMILARITY = 0.65
NEIGHBOR_K = 10
VALIDATION_SEEDS = [2026]  # For a more robust study, set [2026,2027,2028].
ENSEMBLE_MEMBERS = 3
TREES_PER_MEMBER = 80
TEST_FRACTION = 0.2
OUTPUT = Path('cyp_notebook_03_results')
OUTPUT.mkdir(exist_ok=True)

# Local testing override, irrelevant for a normal Colab run.
if os.getenv('CYP_NOTEBOOK_TEST') == '1':
    ENSEMBLE_MEMBERS = 2
    TREES_PER_MEMBER = 8
    VALIDATION_SEEDS = [2026]
    OUTPUT = Path(os.getenv('CYP_NOTEBOOK_OUTPUT', 'cyp_notebook_03_test_results'))
    OUTPUT.mkdir(exist_ok=True)

print('Python:',platform.python_version(), '| RDKit:',rdBase.rdkitVersion)
print('Validation seeds:',VALIDATION_SEEDS,'| Ensemble:',ENSEMBLE_MEMBERS,'×',TREES_PER_MEMBER,'trees')

## 3 · Download training data and verify schema

The dataset is the public [OpenADMET CYP challenge training CSV](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test). The four direct-inhibition pIC50 columns are sparse, and confidence intervals use the suffixes `_conf_low` and `_conf_high`. **Never interpret an absent endpoint as an inactive compound.**

The optional `CYP_DATA_URL` environment variable is only for local execution tests. The default Colab run downloads the official public CSV.

In [ ]:
DATA_URL = os.getenv('CYP_DATA_URL',
    'https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_inhibition.csv')
raw = pd.read_csv(DATA_URL)
print('Input molecules:',len(raw))
required = ['SMILES']
for iso in ISOFORMS:
    required += [f'{iso}_pIC50_direct_inhibition',
                 f'{iso}_pIC50_direct_inhibition_conf_low',
                 f'{iso}_pIC50_direct_inhibition_conf_high']
missing = [col for col in required if col not in raw]
if missing:
    raise ValueError(f'Dataset schema mismatch. Missing: {missing}; actual columns: {list(raw.columns)}')
print('Found all expected molecular and pIC50 confidence-interval columns.')
display(raw.head(3))

## 4 · Standardize molecular identities and construct ECFP4 fingerprints

- Parse SMILES; skip invalid structures.
- Canonicalize molecule identity so duplicate structures cannot independently inflate cliff statistics.
- The official data currently contain unique structures. If duplicates ever appear, this notebook halts rather than silently choosing a measurement or averaging potencies.
- Generate RDKit Morgan/ECFP-like radius-2, 2048-bit fingerprints.

In [ ]:
records=[]
for idx, row in raw.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
    if mol is None:
        continue
    records.append((idx, Chem.MolToSmiles(mol, canonical=True), mol))
clean_indices, canonical, mols = zip(*records) if records else ([], [], [])
work = raw.loc[list(clean_indices)].copy().reset_index(drop=True)
work['canonical_smiles'] = canonical
if work['canonical_smiles'].duplicated().any():
    duplicates = work.loc[work['canonical_smiles'].duplicated(keep=False), 'canonical_smiles'].head(5).tolist()
    raise ValueError('Duplicate canonical molecules found; resolve endpoint replicates explicitly. Examples: '+str(duplicates))

name_col = 'Molecule_Name' if 'Molecule_Name' in work.columns else 'canonical_smiles'
fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [fpg.GetFingerprint(mol) for mol in mols]
X = np.zeros((len(fps),2048),dtype=np.uint8)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp,X[i])

def scaffold_key(mol):
    core = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(core, canonical=True) if core.GetNumAtoms() else 'ACYCLIC_'+Chem.MolToSmiles(mol,canonical=True)
scaffolds = np.asarray([scaffold_key(m) for m in mols],dtype=object)

measurements={}
for iso in ISOFORMS:
    root=f'{iso}_pIC50_direct_inhibition'
    measurements[iso]={
        'y':pd.to_numeric(work[root],errors='coerce').to_numpy(float),
        'lower':pd.to_numeric(work[root+'_conf_low'],errors='coerce').to_numpy(float),
        'upper':pd.to_numeric(work[root+'_conf_high'],errors='coerce').to_numpy(float),
    }
print('Valid unique molecules:',len(work),'| fingerprints:',X.shape)
for iso in ISOFORMS:
    arr=measurements[iso]['y']
    print(f'{iso}: {np.isfinite(arr).sum():,} measured endpoints, {len(arr)-np.isfinite(arr).sum():,} missing')

## 5 · Pairwise similarity and experimental activity differences

For each isoform, form **unordered pairs of compounds both measured for that isoform**. Keep pairs with ECFP4 Tanimoto ≥0.65; the primary activity-cliff threshold is Tanimoto ≥0.75 and |ΔpIC50| ≥1.0 (at least tenfold IC50 change).

The **conservative interval gap** is the minimum possible |ΔpIC50| if each true value could lie anywhere within its reported interval. A *CI-robust* cliff has a minimum gap ≥1.0. This is a deliberately stringent sensitivity definition—not a formal simultaneous 95% statistical test. Missing intervals are **not** coded as non-cliffs.

This phase analyzes all training data retrospectively; it is **not a predictive evaluation**.

In [ ]:
"""Pairwise CYP inhibition activity-cliff utilities.

This module makes no claims that a large measured difference establishes a
physical mechanism. It handles missing experimental intervals conservatively.
"""
from __future__ import annotations

from typing import Iterable
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFMCS
from rdkit.Chem.Draw import MolsToGridImage
from scipy.stats import spearmanr


def conservative_interval_gap(lo_a, hi_a, lo_b, hi_b):
    """Smallest absolute difference compatible with two closed intervals.

    Return NaN if either interval is absent or invalid. If intervals overlap,
    the minimum compatible difference is zero.
    """
    vals = np.asarray([lo_a, hi_a, lo_b, hi_b], dtype=float)
    if not np.isfinite(vals).all() or lo_a > hi_a or lo_b > hi_b:
        return np.nan
    return float(max(0.0, lo_a-hi_b, lo_b-hi_a))


def enumerate_similar_pairs(fps, measured_indices, y, lower, upper, minimum_similarity=.65):
    """Enumerate each eligible unordered pair exactly once.

    DataFrame row indices are global positions in the cleaned molecular table.
    Only pairs with both measured endpoints are eligible.
    """
    measured_indices = np.asarray(measured_indices, dtype=int)
    if len(measured_indices) < 2:
        return pd.DataFrame(columns=[
            'index_a','index_b','similarity','delta_pic50',
            'minimum_interval_gap','valid_both_ci'])
    if len(set(measured_indices.tolist())) != len(measured_indices):
        raise ValueError('measured_indices contains duplicate rows')
    out = []
    subset_fps = [fps[int(i)] for i in measured_indices]
    for ai in range(len(measured_indices)-1):
        gi = int(measured_indices[ai])
        sims = DataStructs.BulkTanimotoSimilarity(subset_fps[ai], subset_fps[ai+1:])
        for bj, sim in enumerate(sims, ai+1):
            if sim < minimum_similarity:
                continue
            gj = int(measured_indices[bj])
            gap = conservative_interval_gap(lower[gi], upper[gi], lower[gj], upper[gj])
            out.append((gi, gj, float(sim), float(abs(y[gi]-y[gj])),
                        gap, bool(np.isfinite(gap))))
    return pd.DataFrame(out, columns=[
        'index_a','index_b','similarity','delta_pic50',
        'minimum_interval_gap','valid_both_ci'])


def characterize_cliffs(pairs, similarity_cutoff=.75, delta_cutoff=1.0):
    """Label estimated and conservative interval-supported activity cliffs."""
    frame = pairs.loc[pairs['similarity'] >= similarity_cutoff].copy()
    frame['is_point_cliff'] = frame['delta_pic50'] >= delta_cutoff
    frame['is_ci_robust_cliff'] = (
        frame['minimum_interval_gap'] >= delta_cutoff
    ) & frame['valid_both_ci']
    return frame


def cliff_prevalence(pairs, n_observed, sim_cutoff=.75, delta_cutoff=1.):
    """Return multiple denominators to avoid misleading pair-count comparisons."""
    eligible = characterize_cliffs(pairs, sim_cutoff, delta_cutoff)
    cliffs = eligible.loc[eligible['is_point_cliff']]
    touched = set(cliffs['index_a'].tolist()) | set(cliffs['index_b'].tolist())
    return {
        'n_observed': int(n_observed),
        'n_possible_pairs': int(n_observed * (n_observed-1) // 2),
        'n_similar_pairs': int(len(eligible)),
        'n_point_cliffs': int(len(cliffs)),
        'n_ci_robust_cliffs': int(eligible['is_ci_robust_cliff'].sum()),
        'n_point_cliffs_with_valid_ci': int(cliffs['valid_both_ci'].sum()) if len(cliffs) else 0,
        'point_cliff_rate_per_similar_pair': float(cliffs.shape[0]/len(eligible)) if len(eligible) else np.nan,
        'robust_cliff_rate_per_similar_pair': float(eligible['is_ci_robust_cliff'].sum()/len(eligible)) if len(eligible) else np.nan,
        'n_compounds_in_cliffs': int(len(touched)),
        'fraction_compounds_in_cliffs': float(len(touched)/n_observed) if n_observed else np.nan,
    }


def neighborhood_train_features(query_fp, train_fps, train_y, k=10,
                                cliff_similarity=.75, delta_cutoff=1.):
    """Compute label-based neighborhood features from TRAINING molecules only.

    The query's experimental y is never used. A rate is undefined (NaN) if
    no similar pairs are present among the training neighbors.
    """
    if len(train_fps) == 0:
        return dict(max_train_similarity=np.nan, train_neighborhood_activity_sd=np.nan,
                    train_neighbor_activity_range=np.nan, local_train_cliff_rate=np.nan,
                    n_local_similar_pairs=0, n_local_cliff_pairs=0)
    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(query_fp, train_fps))
    best = np.argsort(-sims, kind='stable')[:min(k,len(sims))]
    ys = np.asarray(train_y, dtype=float)[best]
    same, cliffs = 0, 0
    for i in range(len(best)-1):
        local_sims = DataStructs.BulkTanimotoSimilarity(train_fps[int(best[i])],
                                                        [train_fps[int(j)] for j in best[i+1:]])
        for offset, similarity in enumerate(local_sims, i+1):
            if similarity >= cliff_similarity:
                same += 1
                if abs(ys[i]-ys[offset]) >= delta_cutoff:
                    cliffs += 1
    return dict(
        max_train_similarity=float(sims[best[0]]),
        train_neighborhood_activity_sd=float(np.std(ys)) if len(ys)>1 else np.nan,
        train_neighbor_activity_range=float(np.ptp(ys)) if len(ys)>1 else np.nan,
        local_train_cliff_rate=(float(cliffs / same) if same else np.nan),
        n_local_similar_pairs=int(same),
        n_local_cliff_pairs=int(cliffs),
    )


def safe_spearman(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 5 or len(np.unique(np.asarray(x)[valid])) < 2:
        return np.nan
    return float(spearmanr(np.asarray(x)[valid], np.asarray(y)[valid]).statistic)


def risk_coverage(y_true, y_hat, uncertainty, fractions=None):
    if fractions is None:
        fractions = np.linspace(.1, 1., 10)
    y_true, y_hat, uncertainty = map(lambda x: np.asarray(x, dtype=float),
                                      (y_true, y_hat, uncertainty))
    finite = np.isfinite(y_true) & np.isfinite(y_hat) & np.isfinite(uncertainty)
    y_true, y_hat, uncertainty = y_true[finite], y_hat[finite], uncertainty[finite]
    if len(y_true)==0:
        return pd.DataFrame(columns=['coverage','n_retained','mae'])
    order = np.argsort(uncertainty, kind='stable')
    rows=[]
    for coverage in fractions:
        n = max(1, int(np.ceil(float(coverage)*len(order))))
        ix = order[:n]
        rows.append(dict(coverage=float(coverage), n_retained=int(n),
                         mae=float(np.mean(abs(y_true[ix]-y_hat[ix])))))
    return pd.DataFrame(rows)


def highlight_noncommon_atoms(mol_a, mol_b, mcs_timeout=2):
    """Return unmatched atom indices using an RDKit MCS with a short timeout.

    Highlighted atoms are *not* necessarily a single chemical transformation.
    """
    try:
        mcs = rdFMCS.FindMCS([mol_a, mol_b], timeout=mcs_timeout,
                             ringMatchesRingOnly=True, completeRingsOnly=True,
                             matchChiralTag=False)
        pattern = Chem.MolFromSmarts(mcs.smartsString) if mcs.numAtoms>0 else None
        if pattern is None:
            return list(range(mol_a.GetNumAtoms())), list(range(mol_b.GetNumAtoms()))
        match_a = set(mol_a.GetSubstructMatch(pattern))
        match_b = set(mol_b.GetSubstructMatch(pattern))
        return ([i for i in range(mol_a.GetNumAtoms()) if i not in match_a],
                [i for i in range(mol_b.GetNumAtoms()) if i not in match_b])
    except Exception:
        return [], []


def render_pair(mol_a, mol_b, legend_a, legend_b, output_png):
    """Save side-by-side molecules highlighting atoms outside a common MCS."""
    a = Chem.Mol(mol_a)
    b = Chem.Mol(mol_b)
    ia, ib = highlight_noncommon_atoms(a,b)
    result = MolsToGridImage([a,b], molsPerRow=2, subImgSize=(380,290),
                             legends=[str(legend_a), str(legend_b)],
                             highlightAtomLists=[ia,ib],useSVG=False,returnPNG=True)
    from pathlib import Path
    if isinstance(result, bytes):
        Path(output_png).write_bytes(result)
    elif hasattr(result, 'data') and isinstance(result.data, bytes):
        Path(output_png).write_bytes(result.data)
    elif hasattr(result, 'save'):
        result.save(str(output_png))
    else:
        raise TypeError(f'Unexpected RDKit image result: {type(result)}')
    return output_png


In [ ]:
all_pairs=[]
scan_start=time.time()
for iso in ISOFORMS:
    p=measurements[iso]
    measured=np.flatnonzero(np.isfinite(p['y']))
    pairs=enumerate_similar_pairs(fps,measured,p['y'],p['lower'],p['upper'],
                                  minimum_similarity=PAIR_COLLECTION_MIN_SIMILARITY)
    pairs.insert(0,'isoform',iso)
    all_pairs.append(pairs)
    print(f'{iso}: {len(measured):,} measured, {len(pairs):,} pairs with similarity ≥{PAIR_COLLECTION_MIN_SIMILARITY:.2f}')
all_pairs_df=pd.concat(all_pairs,ignore_index=True)
print(f'Enumerated {len(all_pairs_df):,} qualifying pairs in {(time.time()-scan_start):.1f} seconds')

## 6 · Cliff frequency across CYP isoforms

In [ ]:
summary_rows=[]
cliff_frames=[]
for iso in ISOFORMS:
    subset=all_pairs_df.loc[all_pairs_df.isoform==iso]
    n=int(np.isfinite(measurements[iso]['y']).sum())
    row=cliff_prevalence(subset,n,SIMILARITY_THRESHOLD,DELTA_PIC50_THRESHOLD)
    row['isoform']=iso
    summary_rows.append(row)
    flags=characterize_cliffs(subset,SIMILARITY_THRESHOLD,DELTA_PIC50_THRESHOLD)
    cliff_frames.append(flags)
summary=pd.DataFrame(summary_rows)
cliff_table=pd.concat(cliff_frames,ignore_index=True)
display(summary.round(4))
fig,axes=plt.subplots(1,2,figsize=(11,4.1))
axes[0].bar(summary['isoform'],summary['point_cliff_rate_per_similar_pair'].fillna(0))
axes[0].set_title('Cliff pairs / eligible similar pairs')
axes[0].set_ylabel('Proportion of similar pairs')
axes[0].set_xlabel('')
axes[1].bar(summary['isoform'],summary['fraction_compounds_in_cliffs'].fillna(0))
axes[1].set_title('Compounds belonging to ≥1 point-estimate cliff')
axes[1].set_ylabel('Fraction of measured compounds')
axes[1].set_xlabel('')
plt.tight_layout()
plt.savefig(OUTPUT/'01_cliff_prevalence.png',dpi=170)
plt.show()

## 7 · Threshold sensitivity and CI-supported differences

Activity-cliff prevalence depends strongly on both thresholds and on how densely different chemical series were sampled. Examine the full sensitivity grid rather than focusing on a single threshold.

CI-robust pair proportions are **lower bounds** on the prevalence of pronounced differences under this conservative interval definition. Interpret cross-CYP differences with unequal endpoint coverage in mind.

In [ ]:
sensitivity=[]
for iso in ISOFORMS:
    subset=all_pairs_df.loc[all_pairs_df.isoform==iso]
    n=int(np.isfinite(measurements[iso]['y']).sum())
    for sim_cutoff in [0.65,0.75,0.85]:
        for delta_cutoff in [0.75,1.0,1.5]:
            r=cliff_prevalence(subset,n,sim_cutoff,delta_cutoff)
            r.update(isoform=iso,similarity_cutoff=sim_cutoff,delta_cutoff=delta_cutoff)
            sensitivity.append(r)
sensitivity_df=pd.DataFrame(sensitivity)
fig,axes=plt.subplots(2,2,figsize=(11,8))
for ax, iso in zip(axes.flat,ISOFORMS):
    tab=sensitivity_df[sensitivity_df.isoform==iso].pivot(
        index='delta_cutoff',columns='similarity_cutoff',values='point_cliff_rate_per_similar_pair')
    values=tab.to_numpy(dtype=float)
    img=ax.imshow(values,aspect='auto',cmap='viridis',vmin=0)
    ax.set_xticks(range(len(tab.columns)),labels=[str(v) for v in tab.columns])
    ax.set_yticks(range(len(tab.index)),labels=[str(v) for v in tab.index])
    for rr in range(values.shape[0]):
        for cc in range(values.shape[1]):
            if np.isfinite(values[rr,cc]):
                ax.text(cc,rr,f'{values[rr,cc]:.3f}',ha='center',va='center',color='white',fontsize=9)
    ax.set_xlabel('Similarity cutoff')
    ax.set_ylabel('ΔpIC50 cutoff')
    ax.set_title(iso)
plt.suptitle('Sensitivity: proportion of similar pairs classified as cliffs',y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT/'02_threshold_sensitivity.png',dpi=170,bbox_inches='tight')
plt.show()

ci_summary=[]
for iso in ISOFORMS:
    subset=cliff_table[(cliff_table.isoform==iso) & cliff_table.is_point_cliff]
    robust=int(subset['is_ci_robust_cliff'].sum())
    with_ci=int(subset['valid_both_ci'].sum())
    ci_summary.append(dict(isoform=iso,n_point_cliffs=len(subset),
        n_point_cliffs_with_valid_ci=with_ci,n_ci_robust_cliffs=robust,
        robust_fraction_among_point_cliffs_with_ci=robust/with_ci if with_ci else np.nan))
ci_summary_df=pd.DataFrame(ci_summary)
display(ci_summary_df)

## 8 · Inspect representative chemical pairs

By default, choose up to three high-separation pairs per CYP isoform, prioritizing conservative CI-robust cliffs and diversity in the molecules displayed. The RDKit maximum-common-substructure algorithm highlights atoms **outside** one detected common substructure. Highlighting is a visual aid, **not proof of a specific transformation or metabolic mechanism**. Maximum-common-substructure search is time-limited.

In [ ]:
selected_pairs=[]
for iso in ISOFORMS:
    sub=cliff_table.loc[(cliff_table.isoform==iso)&cliff_table.is_point_cliff].copy()
    if len(sub)==0:
        print(f'{iso}: no cliff pairs found at primary thresholds')
        continue
    sub=sub.sort_values(['is_ci_robust_cliff','minimum_interval_gap','delta_pic50','similarity'],
                         ascending=[False,False,False,False], na_position='last')
    used=set()
    for _, row in sub.iterrows():
        a,b=int(row.index_a),int(row.index_b)
        if a in used or b in used:
            continue
        selected_pairs.append(row.to_dict())
        used.update([a,b])
        if sum(1 for r in selected_pairs if r['isoform']==iso)>=3:
            break

for order, row in enumerate(selected_pairs,1):
    iso=row['isoform']
    a,b=int(row['index_a']),int(row['index_b'])
    ya=measurements[iso]['y'][a]
    yb=measurements[iso]['y'][b]
    robust='CI-robust' if row['is_ci_robust_cliff'] else 'point-estimate-only'
    legends=[f'{str(work.loc[a,name_col])[:17]} | pIC50 {ya:.2f}',
             f'{str(work.loc[b,name_col])[:17]} | pIC50 {yb:.2f}']
    fig_path=OUTPUT/f'03_pair_{order:02d}_{iso}.png'
    render_pair(mols[a],mols[b],legends[0],legends[1],fig_path)
    print(f'{iso} | Tanimoto={row["similarity"]:.3f} | ΔpIC50={row["delta_pic50"]:.2f} | {robust}')
    display(Image(filename=str(fig_path)))
selected_pairs_df=pd.DataFrame(selected_pairs)

## 9 · Leakage-safe test: do training-neighborhood cliffs predict held-out errors?

**This is the key methodological test.** For each CYP and a scaffold-grouped train/validation split:

1. Fit a small bootstrap random-forest ensemble on **training structures and training pIC50 only**.
2. For every validation molecule, retrieve its ten nearest **training** neighbors using ECFP4/Tanimoto.
3. From those training neighbors' labels alone, calculate:
   - activity standard deviation;
   - activity range;
   - fraction of mutually similar training-neighbor pairs that form measured activity cliffs.
4. Compare these training-derived features, nearest-neighbor similarity and model ensemble disagreement against the held-out absolute error.

**Important:** The local cliff *rate* is undefined (NaN) if there are no qualifying training-neighbor pairs. Report its sample coverage; never substitute zero. No validation pIC50 or confidence interval is used to derive a predictive uncertainty feature.

This single-seed experiment is illustrative. To strengthen claims, set `VALIDATION_SEEDS` to three or five values and report variability.

In [ ]:
valid_predictions=[]
start=time.time()
for iso in ISOFORMS:
    p=measurements[iso]
    y=p['y']
    observed=np.flatnonzero(np.isfinite(y))
    observed_groups=scaffolds[observed]
    for seed in VALIDATION_SEEDS:
        gss=GroupShuffleSplit(n_splits=1,test_size=TEST_FRACTION,random_state=seed)
        local_train,local_val=next(gss.split(observed,y[observed],groups=observed_groups))
        train_idx=observed[local_train]
        val_idx=observed[local_val]
        assert set(scaffolds[train_idx]).isdisjoint(set(scaffolds[val_idx]))
        if len(train_idx)<20 or len(val_idx)<5:
            raise ValueError(f'Too few measured compounds for {iso}; inspect group split.')
        rng=np.random.default_rng(seed)
        member_preds=[]
        for rep in range(ENSEMBLE_MEMBERS):
            sampled=rng.choice(train_idx,size=len(train_idx),replace=True)
            model=RandomForestRegressor(n_estimators=TREES_PER_MEMBER,
                max_features=.7,min_samples_leaf=2,n_jobs=-1,random_state=seed+rep)
            model.fit(X[sampled],y[sampled])
            member_preds.append(model.predict(X[val_idx]))
        arr=np.asarray(member_preds,dtype=float)
        y_hat=arr.mean(axis=0)
        model_unc=arr.std(axis=0,ddof=1) if len(arr)>1 else np.zeros(len(val_idx))
        tr_fps=[fps[i] for i in train_idx]
        tr_y=y[train_idx]
        for j, vi in enumerate(val_idx):
            local=neighborhood_train_features(fps[int(vi)],tr_fps,tr_y,k=NEIGHBOR_K,
                        cliff_similarity=SIMILARITY_THRESHOLD,
                        delta_cutoff=DELTA_PIC50_THRESHOLD)
            valid_predictions.append(dict(
                isoform=iso,seed=seed,row_index=int(vi),
                molecule_name=str(work.loc[int(vi),name_col]),
                true_pic50=float(y[int(vi)]),pred_pic50=float(y_hat[j]),
                abs_error=float(abs(y[int(vi)]-y_hat[j])),
                signed_error=float(y_hat[j]-y[int(vi)]),
                ensemble_std=float(model_unc[j]),
                **local
            ))
    print(iso,': finished',len(VALIDATION_SEEDS),'scaffold split(s)')
validation_df=pd.DataFrame(valid_predictions)
validation_df['similarity_uncertainty']=1.0-validation_df['max_train_similarity']
print('Elapsed minutes:',round((time.time()-start)/60,2))
print('Total held-out predictions:',len(validation_df))
display(validation_df.head())

## 10 · Compare training-only uncertainty signals

The Spearman coefficient should be **positive** if a larger uncertainty signal tends to correspond to a larger absolute error. Raw pair counts are not the unit of analysis here: each row corresponds to a held-out compound prediction. If using repeated seeds, the same compound may appear more than once, so do not treat those rows as independent replicates for formal p-values.

In [ ]:
SIGNALS={
    'ensemble_std':'Ensemble disagreement',
    'similarity_uncertainty':'1 − max training similarity',
    'train_neighborhood_activity_sd':'Training-neighbor activity SD',
    'train_neighbor_activity_range':'Training-neighbor activity range',
    'local_train_cliff_rate':'Training-neighbor cliff rate',
}
assoc=[]
for (iso,seed), group in validation_df.groupby(['isoform','seed']):
    for col,label in SIGNALS.items():
        ok=np.isfinite(group[col]) & np.isfinite(group['abs_error'])
        assoc.append(dict(isoform=iso,seed=seed,feature=col,signal_label=label,
            coverage=float(ok.mean()),n_valid=int(ok.sum()),
            spearman_rho=safe_spearman(group[col].to_numpy(),group['abs_error'].to_numpy())))
assoc_df=pd.DataFrame(assoc)
assoc_summary=assoc_df.groupby(['isoform','feature','signal_label'],as_index=False).agg(
    mean_rho=('spearman_rho','mean'),sd_rho=('spearman_rho','std'),
    mean_coverage=('coverage','mean'),mean_n_valid=('n_valid','mean'))
display(assoc_summary.round(3))
fig,axes=plt.subplots(2,2,figsize=(12,8),sharey=True)
for ax,iso in zip(axes.flat,ISOFORMS):
    part=assoc_summary[assoc_summary.isoform==iso].set_index('feature')
    vals=[part.loc[s,'mean_rho'] if s in part.index else np.nan for s in SIGNALS]
    ax.bar(range(len(SIGNALS)),vals)
    ax.axhline(0,color='black',linewidth=.8)
    ax.set_xticks(range(len(SIGNALS)))
    ax.set_xticklabels(['Ensemble SD','Chemical distance','Neighbor SD','Neighbor range','Cliff rate'],
                        rotation=40,ha='right')
    ax.set_title(iso)
    ax.set_ylabel('Spearman ρ with absolute error')
fig.suptitle('Prospective training-only uncertainty signals (scaffold splits)',y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT/'04_train_only_uncertainty_association.png',dpi=170,bbox_inches='tight')
plt.show()

## 11 · Risk–coverage comparison

An uncertainty signal is useful for selective prediction if retaining the **lowest**-uncertainty compounds leads to lower error. Curves are conditioned on rows where each signal exists; for cliff rate this subset can be much smaller. **Do not compare absolute MAE across signals with different sample coverage without examining the common subset.**

The second plot compares all signals using the same per-isoform complete-case subset, where every score is defined. It may have very low coverage and is therefore a sensitivity analysis rather than a default benchmark.

In [ ]:
risk_rows=[]
for (iso,seed),g in validation_df.groupby(['isoform','seed']):
    for sig in SIGNALS:
        curve=risk_coverage(g.true_pic50,g.pred_pic50,g[sig])
        curve['isoform']=iso
        curve['seed']=seed
        curve['signal']=sig
        risk_rows.append(curve)
risk_df=pd.concat(risk_rows,ignore_index=True)
fig,axes=plt.subplots(2,2,figsize=(12,8),sharey=False)
for ax,iso in zip(axes.flat,ISOFORMS):
    part=risk_df[risk_df.isoform==iso]
    for sig in SIGNALS:
        sub=part[part.signal==sig]
        if len(sub):
            g=sub.groupby('coverage',as_index=False)['mae'].mean()
            ax.plot(g.coverage,g.mae,label=SIGNALS[sig])
    ax.set_title(iso)
    ax.set_xlabel('Fraction retained')
    ax.set_ylabel('MAE, pIC50 units')
axes[0,0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT/'05_risk_coverage_raw.png',dpi=170)
plt.show()

complete_case=[]
for (iso,seed),g in validation_df.groupby(['isoform','seed']):
    mask=g[list(SIGNALS)].notna().all(axis=1)
    gc=g.loc[mask]
    if len(gc)<10:
        continue
    for sig in SIGNALS:
        curve=risk_coverage(gc.true_pic50,gc.pred_pic50,gc[sig])
        curve['isoform']=iso
        curve['seed']=seed
        curve['signal']=sig
        curve['n_complete_case']=len(gc)
        complete_case.append(curve)
complete_risk_df=pd.concat(complete_case,ignore_index=True) if complete_case else pd.DataFrame()
print('Complete-case sample counts by isoform and seed:')
display(validation_df.assign(all_scores_available=validation_df[list(SIGNALS)].notna().all(axis=1))
       .groupby(['isoform','seed'])['all_scores_available'].agg(['sum','count']))
if len(complete_risk_df):
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for ax,iso in zip(axes.flat,ISOFORMS):
        sub=complete_risk_df[complete_risk_df.isoform==iso]
        for sig in SIGNALS:
            s=sub[sub.signal==sig]
            if len(s):
                curve=s.groupby('coverage',as_index=False)['mae'].mean()
                ax.plot(curve.coverage,curve.mae,label=SIGNALS[sig])
        ax.set_title(iso)
        ax.set_xlabel('Fraction retained (common subset)')
        ax.set_ylabel('MAE, pIC50 units')
    axes[0,0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT/'06_risk_coverage_common_subset.png',dpi=170)
    plt.show()
else:
    print('No isoform has ≥10 complete-case held-out predictions; common-subset risk analysis skipped.')

## 12 · Diagnostic: activity extremes and interval widths

This final analysis revisits Notebook 02: does error concentrate at activity extremes even after controlling for local chemical neighborhood? The validation target and confidence intervals are used **only for retrospective evaluation** here, never as model inputs.

In [ ]:
ci_records=[]
for r in validation_df.itertuples(index=False):
    interval=measurements[r.isoform]
    lo=interval['lower'][r.row_index]
    hi=interval['upper'][r.row_index]
    ci_records.append(dict(isoform=r.isoform,seed=r.seed,row_index=r.row_index,
        true_pic50=r.true_pic50,pred_pic50=r.pred_pic50,abs_error=r.abs_error,
        ci_width=hi-lo if np.isfinite(lo) and np.isfinite(hi) and hi>=lo else np.nan,
        excess_beyond_ci=conservative_interval_gap(r.pred_pic50,r.pred_pic50,lo,hi)))
ci_df=pd.DataFrame(ci_records)
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for iso, part in ci_df.groupby('isoform'):
    axes[0].scatter(part['true_pic50'],part['abs_error'],s=15,alpha=.4,label=iso)
axes[0].legend()
axes[0].set_title('Prediction error across activity range')
for iso, part in ci_df.groupby('isoform'):
    axes[1].scatter(part['ci_width'],part['abs_error'],s=15,alpha=.4,label=iso)
axes[1].set_title('Prediction error vs experimental CI width')
plt.tight_layout()
plt.savefig(OUTPUT/'07_error_vs_activity_and_ci.png',dpi=170)
plt.show()

ci_summary=ci_df.groupby('isoform',as_index=False).agg(
    n=('abs_error','size'),mean_mae=('abs_error','mean'),
    median_ci_width=('ci_width','median'),
    mean_excess_beyond_ci=('excess_beyond_ci','mean'))
display(ci_summary.round(3))

## 13 · Export tables, figures, metadata, and one ZIP

Exports include the individual qualifying molecular pairs, but **not** the raw training dataset or model weights. Pair tables contain indices referencing the public dataset plus experimental outcomes. Review dataset licensing and competition rules before republishing derived measurements or tables publicly. Do not commit the pair-level CSVs to GitHub without checking whether redistribution is permitted and scientifically appropriate.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

frames={
    'pairwise_similarity_min065.csv':all_pairs_df,
    'primary_threshold_pairs.csv':cliff_table,
    'activity_cliff_summary.csv':summary,
    'threshold_sensitivity.csv':sensitivity_df,
    'confidence_interval_cliff_summary.csv':ci_summary_df,
    'selected_molecular_pairs.csv':selected_pairs_df,
    'validation_predictions_train_only_features.csv':validation_df,
    'uncertainty_associations.csv':assoc_df,
    'uncertainty_association_summary.csv':assoc_summary,
    'risk_coverage_by_signal.csv':risk_df,
    'risk_coverage_common_subset.csv':complete_risk_df,
    'validation_confidence_interval_diagnostics.csv':ci_df,
    'validation_ci_summary.csv':ci_summary,
}
for filename, frame in frames.items():
    frame.to_csv(OUTPUT/filename,index=False)
meta={
    'notebook':'03_activity_cliffs_molecular_similarity.ipynb',
    'dataset_url':DATA_URL,
    'n_input':len(raw),'n_clean_unique':len(work),
    'isoforms':ISOFORMS,'fingerprint':'Morgan radius 2, 2048 bits',
    'primary_similarity_threshold':SIMILARITY_THRESHOLD,
    'primary_delta_pic50_threshold':DELTA_PIC50_THRESHOLD,
    'minimum_pair_similarity_collected':PAIR_COLLECTION_MIN_SIMILARITY,
    'train_neighborhood_k':NEIGHBOR_K,'validation_seeds':VALIDATION_SEEDS,
    'ensemble_members':ENSEMBLE_MEMBERS,'trees_per_member':TREES_PER_MEMBER,
    'scaffold_split':'GroupShuffleSplit on Murcko scaffold; acyclic groups by canonical SMILES',
    'note':'CI gap is a conservative interval-distance diagnostic, not a formal significance test or official metric.',
    'python':platform.python_version(),'rdkit':rdBase.rdkitVersion,
    'sklearn':__import__('sklearn').__version__,
}
(OUTPUT/'run_metadata.json').write_text(json.dumps(meta,indent=2,default=str))
zip_file=Path('cyp_notebook_03_results.zip')
with ZipFile(zip_file,'w',ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT.glob('*')):
        if p.is_file():
            zf.write(p,arcname=p.name)
print(f'Saved {len(list(OUTPUT.glob("*")))} results files in {OUTPUT}/')
print('Results archive:',zip_file, f'({zip_file.stat().st_size/1024:.1f} KiB)')
try:
    from google.colab import files
    files.download(str(zip_file))
except ImportError:
    print('Outside Colab: download the ZIP from the working directory.')

## 14 · How to interpret (and not overinterpret) the results

- **Prevalence:** Compare cliffs per *similar pair* and compounds involved; avoid comparing raw numbers without denominators. The pairs are correlated and depend on library composition.
- **CI robustness:** A pair that fails the conservative gap criterion may still be a genuine cliff; uncertainty may simply be too broad to certify the difference.
- **Chemical interpretation:** Nonmatching atoms in a maximum-common-substructure picture identify changes in chemical graph representation, not mechanisms of CYP binding or metabolism.
- **Prospective association:** Training-neighbor SD, range and cliff rate use training activities only. Compare their associations with held-out errors to chemical distance and ensemble disagreement.
- **Missingness:** The local cliff rate is undefined when no similar pairs exist within the chosen training neighborhood. Its coverage must be reported explicitly.
- **External validity:** A single scaffold split, small ensemble, and shared source dataset are insufficient for publication-level generalization. Repeat seeds and compare different model classes before strong claims.

**Follow-up experiment:** If training-only neighborhood activity variability reliably forecasts errors, test this with LightGBM and a Tanimoto-weighted kNN predictor, then repeat across multiple clustered chemical-series splits.

**Data attribution:** [OpenADMET dataset](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test) · [Challenge tutorial](https://github.com/OpenADMET/CYP-Challenge-Tutorial) · [Challenge announcement](https://openadmet.ghost.io/announcing-openadmets-cyp-inhibition-blind-challenge/).